# RAG Is Simpler Than You Think

- Six ways to search the same 14 photos, cheapest first
- One Pixeltable database holds the photos, captions, embeddings, questions, and answers
- The questions are text. The evidence is images.
- Add the next recipe only when the one before it misses

Inspired by Rafael Pierre's [RAG Is Simpler Than You Think](https://www.lighthousenewsletter.com/p/rag-is-simpler-than-you-think).

## Setup

- Run this notebook from `simpler-rag/`, next to the recipe files
- Recipes 2 and 6 call OpenAI: set `OPENAI_API_KEY` where the `pxt` daemon runs

In [ ]:
!uv sync --package simpler-rag

## Recipe 1: search titles

- A `photos` table: ID, image, title
- Keyword search. No models, no API calls.
- `contains()` is a substring match, not ranked full-text search

In [ ]:
!cat recipe1_store.py

In [ ]:
!uv run pxt schema diff recipe1_store.py rag_demo

In [ ]:
!uv run pxt schema update recipe1_store.py rag_demo -f

In [ ]:
import pixeltable as pxt

BASE = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/images/'
PHOTOS = [  # (photo_id, image number, title)
    (1, 1, 'Taxi on a city street'),
    (2, 9, 'Bento lunch box'),
    (3, 16, 'Baseball batter'),
    (4, 25, 'Giraffe at the zoo'),
    (5, 30, 'Vase of flowers'),
    (6, 34, 'Zebra grazing'),
    (7, 42, 'Dog asleep on the shoes'),
    (8, 49, 'Horse riders performing'),
    (9, 57, 'Tennis match'),
    (10, 61, 'Elephant trek in the jungle'),
    (11, 64, 'Street clock'),
    (12, 71, 'Commuter train in the yard'),
    (13, 80, 'Zebra in an enclosure'),
    (14, 90, 'Cow in a pasture'),
]

photos = pxt.get_table('rag_demo.photos')
photos.insert(
    [{'photo_id': i, 'image': f'{BASE}{n:012d}.jpg', 'title': t} for i, n, t in PHOTOS]
)

In [ ]:
photos.where(photos.title.contains('zebra', case=False)).select(
    photos.photo_id, photos.image, photos.title
).collect()

In [ ]:
# "what are the big animals with really long necks?"
photos.where(photos.title.contains('long neck', case=False)).select(
    photos.photo_id, photos.image, photos.title
).collect()

- ✅ `zebra` finds both zebra photos
- ❌ `long neck` finds nothing. The giraffe's title says `Giraffe`.

## Recipe 2: rewrite the question

- A `queries` table: every question is a row
- An LLM rewrites each question into one search word
- `title_hits` is a query stored as a column. It runs on every insert.
- Still no embeddings. If results miss, change the prompt.

In [ ]:
!cat recipe2_rewrite.py

In [ ]:
!uv run pxt schema diff recipe2_rewrite.py rag_demo

In [ ]:
!uv run pxt schema update recipe2_rewrite.py rag_demo -f

In [ ]:
queries = pxt.get_table('rag_demo.queries')
queries.insert([{'raw': 'what are the big animals with really long necks?'}])
queries.select(queries.raw, queries.search_query, queries.title_hits).collect()

- ✅ The question becomes `giraffe`, and the title search finds photo 4
- ❌ Titles are short. Most of what a photo shows isn't in its title.

## Recipe 3: search captions

- A BLIP model writes a caption for every photo
- A computed column: it fills the 14 photos now, and new photos on insert
- The model downloads on the first run

In [ ]:
!cat recipe3_caption.py

In [ ]:
!uv run pxt schema diff recipe3_caption.py rag_demo

In [ ]:
!uv run pxt schema update recipe3_caption.py rag_demo -f

In [ ]:
photos = pxt.get_table('rag_demo.photos')  # reload to see the new column
photos.select(photos.photo_id, photos.image, photos.title, photos.caption).collect()

In [ ]:
photos.where(photos.caption.contains('field')).select(
    photos.photo_id, photos.image, photos.title, photos.caption
).collect()

In [ ]:
photos.where(photos.caption.contains('giraffe')).select(
    photos.photo_id, photos.image, photos.title, photos.caption
).collect()

- ✅ `field` finds the grazing zebra and the cow. Their titles say `grazing` and `pasture`.
- ⚠️ Captions are model output. Photo 4's caption says `gife`, so `giraffe` finds nothing.

## Recipe 4: search the images

- A CLIP embedding index on the `image` column
- CLIP puts images and text in one space, so a text question can find a photo
- The index stays current on insert. Nothing to chunk: one photo is one item.

In [ ]:
!cat recipe4_search.py

In [ ]:
!uv run pxt schema diff recipe4_search.py rag_demo

In [ ]:
!uv run pxt schema update recipe4_search.py rag_demo -f

In [ ]:
photos = pxt.get_table('rag_demo.photos')  # reload to see the index
sim = photos.image.similarity(string='a striped animal', idx='clip_idx')
photos.order_by(sim, asc=False).limit(3).select(
    photos.photo_id, photos.image, photos.title, photos.caption, score=sim
).collect()

### Keywords and embeddings together

- Filter by a word, then rank what's left by meaning

In [ ]:
sim = photos.image.similarity(string='a jungle', idx='clip_idx')
photos.where(photos.caption.contains('riding')).order_by(sim, asc=False).select(
    photos.photo_id, photos.image, photos.title, photos.caption, score=sim
).collect()

- ✅ `a striped animal` ranks both zebras first. No title or caption says `striped`.
- ✅ `riding` plus `a jungle` puts the elephant trek above the horse riders

## Recipe 5: swap the embedding model

- Add a second CLIP model next to the first. Both indexes stay current.
- Compare them on the same question
- Keep the better one and drop the old index

In [ ]:
!cat recipe5_compare.py

In [ ]:
!uv run pxt schema diff recipe5_compare.py rag_demo

In [ ]:
!uv run pxt schema update recipe5_compare.py rag_demo -f

In [ ]:
photos = pxt.get_table('rag_demo.photos')  # reload to see the new index

def top3(query, idx):
    sim = photos.image.similarity(string=query, idx=idx)
    return photos.order_by(sim, asc=False).limit(3).select(
        photos.photo_id, photos.image, photos.title, photos.caption, score=sim
    ).collect()

print('clip_idx (ViT-B/32)')
display(top3('a racket sport', 'clip_idx'))
print('clip_idx_v2 (ViT-B/16)')
display(top3('a racket sport', 'clip_idx_v2'))

- ✅ ViT-B/16 ranks the tennis match first
- ❌ ViT-B/32 ranks a dog asleep on shoes first, just ahead of the tennis match

### New photos

- Insert two photos with no title
- Pixeltable captions them and adds them to both indexes

In [ ]:
photos.insert(
    [
        {'photo_id': 15, 'image': f'{BASE}{36:012d}.jpg'},
        {'photo_id': 16, 'image': f'{BASE}{63:012d}.jpg'},
    ]
)
top3('an umbrella', 'clip_idx_v2')

### Keep the winner

- `recipe5_cutover.py` keeps only `clip_idx_v2`
- The diff marks the drop `DESTRUCTIVE`. The update needs `--allow-destructive`.

In [ ]:
!uv run pxt schema diff recipe5_cutover.py rag_demo

In [ ]:
!uv run pxt schema update recipe5_cutover.py rag_demo --allow-destructive -f

- ✅ New model in, old model out. Two schema updates, no reloading the photos.

## Recipe 6: answer from the photos

- `photo_hits`: a query column that returns the top 3 photos for each question
- `answer`: a vision model reads those photos and cites the photo IDs it used
- New columns fill in for questions already in the table

In [ ]:
!cat recipe6_answer.py

In [ ]:
!uv run pxt schema diff recipe6_answer.py rag_demo

In [ ]:
!uv run pxt schema update recipe6_answer.py rag_demo -f

In [ ]:
queries = pxt.get_table('rag_demo.queries')  # reload to see the new columns
queries.select(queries.raw, queries.search_query, queries.photo_hits, queries.answer).collect()

### Ask a new question

- One insert runs the whole chain: rewrite, search, retrieve, answer

In [ ]:
queries.insert([{'raw': 'which photo shows a sport played with a racket?'}])
queries.select(queries.raw, queries.search_query, queries.photo_hits, queries.answer).collect()

- ✅ Each answer sits next to the question, the rewrite, and the photos it used

## Choose a recipe

- Start with Recipe 1. Stop when your questions get good results.
- Misses because of wording: rewrite the question (Recipe 2)
- Misses because the words aren't in the data: captions or embeddings (Recipes 3 and 4)
- Each recipe is a column or an index on the same tables

To start over, drop `rag_demo` and run the notebook again:

In [ ]:
# pxt.drop_table('rag_demo.queries', if_not_exists='ignore')  # drop first: it queries photos
# pxt.drop_dir('rag_demo', force=True)